# Phenotype residualization

Input: round 2's ancestry-filtered keep-list (person IDs) and a phenotype list TSV. Output: one `FID IID Y` file per phenotype x transform x covariate-set combination, matching `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected format.

Protocol (order matters, see Kemper et al. 2021 Methods):
1. residualize `phenotype ~ covariates`
2. trim residuals > 5 SD from the mean
3. standardize surviving residuals to mean 0, var 1, within each sex

Covariates: sex-at-birth + age always included. PCs / zip3 (factor) / SES vars (`zip3_ses_map`) are each independently toggleable — every combination gets run, not just one chosen set.

## Setup

In [ ]:
library(dplyr)
library(tidyr)
library(readr)
library(stringr)
library(e1071)   # skewness()
library(bigrquery)

## Inputs

- `KEEP_LIST_PATH`: round 2 output — person IDs passing ancestry filtering.
- `PHENO_LIST_PATH`: TSV describing which phenotypes to pull (see schema below).
- `PC_PATH`: round 1's PCA output (`.eigenvec`), or round 1's AoU-projected `.sscore` once that step is done.
- `ZIP3_SES_TABLE`: AoU's `zip3_ses_map` — exact join path (likely via `observation`, not directly on `person`) still needs confirming against the real workbench, flagged below.

Phenotype list TSV schema (`PHENO_LIST_PATH`):

| column | purpose |
|---|---|
| `phenotype_name` | label used in output filenames/diagnostics |
| `source` | `measurement` / `survey` / `condition` -- picks the extraction path |
| `concept_id` | AoU concept ID (comma-separated for multiple) |
| `value_field` | optional -- which field holds the numeric value |
| `notes` | free text, not used programmatically |

In [ ]:
KEEP_LIST_PATH <- "PLACEHOLDER"   # round 2 output, one person_id per line
PHENO_LIST_PATH <- "PLACEHOLDER"  # phenotype list TSV, see schema above
PC_PATH <- "PLACEHOLDER"          # round 1 .eigenvec (or AoU-projected .sscore)
OUT_DIR <- "PLACEHOLDER"          # where the FID/IID/Y files get written

dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

## Read the ID list and phenotype list

`keep_ids`: everyone downstream gets restricted to this set before anything else -- residualization, transforms, and diagnostics are all computed within this cohort only, not the full AoU population.

In [ ]:
keep_ids <- read_lines(KEEP_LIST_PATH)

pheno_list <- read_tsv(PHENO_LIST_PATH, col_types = cols(.default = "c"))
stopifnot(all(c("phenotype_name", "source", "concept_id") %in% names(pheno_list)))
pheno_list

## Pull phenotype + covariate table

**Unresolved, needs the real workbench to pin down:**
- exact BigQuery table/column names per `source` type (`measurement` vs `survey` vs `condition`)
- `zip3_ses_map` join path
- age definition -- age at the specific measurement/response (more correct, varies per phenotype) vs. a single fixed age pulled once. Written here as per-phenotype since that's the more defensible default; simplify later if it turns out not to matter.

Everything below this cell (transform, covariate-set configs, residualization, export, diagnostics) doesn't depend on these specifics and is fully generic once `pull_phenotype()` returns a table shaped `person_id, phenotype, age, sex_at_birth`.

In [ ]:
pull_phenotype <- function(row, keep_ids) {
  # TODO: real BigQuery query, branching on row$source ("measurement" / "survey" / "condition").
  # Must return: person_id, phenotype (numeric), age (numeric, at the phenotype's own
  # measurement/response time), sex_at_birth (factor) -- restricted to keep_ids.
  stop("pull_phenotype() not implemented -- fill in against the real workbench")
}

pull_covariates <- function(keep_ids, pc_path, zip3_ses_table = NULL) {
  pcs <- read_table(pc_path, col_types = cols(.default = "d", IID = "c")) %>%
    rename(person_id = IID) %>%
    filter(person_id %in% keep_ids)

  zip3 <- tibble(person_id = character(), zip3 = character())        # TODO: real join
  ses <- tibble(person_id = character(), median_income = double())    # TODO: real join

  list(pcs = pcs, zip3 = zip3, ses = ses)
}

## Optional skew-reducing transform

Rank-based inverse-normal transform -- chosen over log/Box-Cox since it doesn't assume a particular skew direction or require positive values, so it works uniformly across an arbitrary phenotype list without per-phenotype tuning. Both `raw` and `invnorm` variants get carried through the rest of the pipeline; skewness before/after is reported so it's clear whether the transform actually helped for each phenotype, rather than applying it blindly.

In [ ]:
inverse_normal_transform <- function(x) {
  r <- rank(x, na.last = "keep", ties.method = "average")
  n <- sum(!is.na(x))
  qnorm((r - 0.5) / n)
}

add_transformed_variant <- function(df, pheno_col) {
  df[[paste0(pheno_col, "__invnorm")]] <- inverse_normal_transform(df[[pheno_col]])
  df
}

skew_summary <- function(x, label) {
  tibble(variant = label, n = sum(!is.na(x)), skewness = skewness(x, na.rm = TRUE))
}

## Covariate-set configs

Named list of formula RHS strings. `sex_at_birth` is handled separately (it's the stratification variable for step 3, not a residualization covariate -- see `residualize_phenotype()` below), so it's not in these formulas.

In [ ]:
pc_cols <- paste0("PC", 1:20)   # matches round 1's 20-PC output

covariate_sets <- list(
  base              = c("age"),
  base_pcs          = c("age", pc_cols),
  base_pcs_zip3     = c("age", pc_cols, "zip3"),
  base_pcs_ses      = c("age", pc_cols, "median_income", "poverty", "deprivation_index"),
  base_pcs_zip3_ses = c("age", pc_cols, "zip3", "median_income", "poverty", "deprivation_index")
)

## Residualize

Ported from the pre-existing `normalize_phenotype()` (was in `02_phenotype/scripts/aou/normalize_phenotype.R` before the repo got stripped back to a skeleton -- logic unchanged, just renamed for clarity against this notebook's terminology).

In [ ]:
residualize_phenotype <- function(df, pheno_col, sex_col, covariate_cols, outlier_sd = 5) {
  formula <- as.formula(paste(pheno_col, "~", paste(covariate_cols, collapse = " + ")))
  fit <- lm(formula, data = df, na.action = na.exclude)
  df$.residual <- residuals(fit)

  keep <- !is.na(df$.residual)
  m <- mean(df$.residual[keep])
  s <- sd(df$.residual[keep])
  not_outlier <- keep & abs(df$.residual - m) <= outlier_sd * s

  df$phenotype_norm <- NA_real_
  for (s_level in unique(df[[sex_col]][not_outlier])) {
    idx <- not_outlier & df[[sex_col]] == s_level
    r <- df$.residual[idx]
    df$phenotype_norm[idx] <- (r - mean(r)) / sd(r)
  }

  list(
    data = df %>% select(-.residual),
    n_input = nrow(df),
    n_retained = sum(!is.na(df$phenotype_norm)),
    r_squared = summary(fit)$r.squared
  )
}

## Main loop

Cross product of {phenotype} x {raw, invnorm} x {covariate-set}, all exported -- no curation, every combination in `pheno_list` x `covariate_sets` gets written out. `write_grm_pheno()` matches `GRM-pairs/full_grm_bin/prep_pheno.R`'s expected `FID IID Y` format.

In [ ]:
write_grm_pheno <- function(df, file) {
  out <- df %>%
    filter(!is.na(phenotype_norm)) %>%
    transmute(FID = person_id, IID = person_id, Y = phenotype_norm)
  write.table(out, file = file, quote = FALSE, sep = " ",
              row.names = FALSE, col.names = TRUE, na = "NA")
}

skew_diagnostics <- list()
combo_diagnostics <- list()

for (i in seq_len(nrow(pheno_list))) {
  row <- pheno_list[i, ]
  name <- row$phenotype_name

  pheno_df <- pull_phenotype(row, keep_ids)
  covars <- pull_covariates(keep_ids, PC_PATH)

  df <- pheno_df %>%
    left_join(covars$pcs, by = "person_id") %>%
    left_join(covars$zip3, by = "person_id") %>%
    left_join(covars$ses, by = "person_id") %>%
    add_transformed_variant("phenotype")

  skew_diagnostics[[paste0(name, "__raw")]] <-
    skew_summary(df$phenotype, "raw") %>% mutate(phenotype = name, .before = 1)
  skew_diagnostics[[paste0(name, "__invnorm")]] <-
    skew_summary(df[["phenotype__invnorm"]], "invnorm") %>% mutate(phenotype = name, .before = 1)

  for (variant_col in c("phenotype", "phenotype__invnorm")) {
    variant_label <- ifelse(variant_col == "phenotype", "raw", "invnorm")

    for (covset_name in names(covariate_sets)) {
      result <- residualize_phenotype(df, variant_col, "sex_at_birth", covariate_sets[[covset_name]])

      out_name <- sprintf("%s__%s__%s", name, variant_label, covset_name)
      write_grm_pheno(result$data, file.path(OUT_DIR, paste0(out_name, ".pheno")))

      combo_diagnostics[[out_name]] <- tibble(
        combo = out_name, phenotype = name, variant = variant_label,
        covariate_set = covset_name,
        n_input = result$n_input, n_retained = result$n_retained,
        r_squared = result$r_squared
      )
    }
  }
}

## Diagnostics summary

In [ ]:
skew_summary_table <- bind_rows(skew_diagnostics)
skew_summary_table   # per phenotype: skewness before/after the invnorm transform

combo_summary_table <- bind_rows(combo_diagnostics)
combo_summary_table  # per phenotype x variant x covariate-set: N retained, R^2